In [9]:
# read and compare files
import re
import pandas as pd
import csv

filepath = "../../results/ena_upload/"
runs_file = filepath + "run-files-2026-08-28T10_28_38.csv"
samples_file = filepath + "samples-2026-08-28T10_28_19.csv"
unsubmitted_file = filepath + "unsubmitted-files-2026-08-28T10_27_57.csv"

def read_ena_csv(path):
    with open(path) as f:
        lines = f.readlines()
    header = next(csv.reader([lines[0]]))
    rows = []
    for line in lines[1:]:
        line = line.strip()
        if line.startswith('"') and line.endswith('"'):
            line = line[1:-1].replace('""', '"')
        rows.append(next(csv.reader([line])))
    return pd.DataFrame(rows, columns=header)

runs = read_ena_csv(runs_file)
samples = read_ena_csv(samples_file)
unsubmitted = read_ena_csv(unsubmitted_file)

# sample ID = part before first "-", leading zeros stripped
def sample_key(name):
    prefix = name.split("-")[0]
    return prefix.lstrip("0") or "0"

# strip path + _R1/_R2.fastq.gz to get base sample name
def get_base_match(filename):
    filename = filename.split("/")[-1]
    match = re.search(r"_(R[12])\.fastq\.gz$", filename)
    return (filename[:match.start()], match.group(1)) if match else (filename, None)

runs["base"], runs["mate"] = zip(*runs["fileName"].map(get_base_match))
unsubmitted["base"], unsubmitted["mate"] = zip(*unsubmitted["fileName"].map(get_base_match))

runs["key"] = runs["base"].map(sample_key)
unsubmitted["key"] = unsubmitted["base"].map(sample_key)
samples["key"] = samples["alias"].map(sample_key)

r1 = set(runs.loc[runs.mate == "R1", "key"])
r2 = set(runs.loc[runs.mate == "R2", "key"])
unsub = set(unsubmitted["key"])

rows = []
for _, srow in samples.iterrows():
    key = srow["key"]
    has_r1, has_r2 = key in r1, key in r2
    rows.append({
        "alias": srow["alias"],
        "key": key,
        "R1": has_r1,
        "R2": has_r2,
        "unsubmitted": key in unsub,
    })

report = pd.DataFrame(rows)

good = report[report.R1 & report.R2 & ~report.unsubmitted]
no_run = report[~report.R1 & ~report.R2]
incomplete = report[report.R1 != report.R2]
flagged = report[report.unsubmitted]

print(f"samples: {len(samples)} | good: {len(good)} | no run: {len(no_run)} | incomplete: {len(incomplete)} | unsubmitted flagged: {len(flagged)}")

# orphans: files that don't match any sample key
orphans = (r1 | r2) - set(samples["key"])
print("orphan run keys (no matching sample):", orphans)

print(flagged.merge(report[["key"]], on="key")[["alias","R1","R2","unsubmitted"]])

samples: 89 | good: 62 | no run: 27 | incomplete: 0 | unsubmitted flagged: 5
orphan run keys (no matching sample): set()
                                        alias     R1     R2  unsubmitted
0  039Ap-CMPN001_49_260114_LH00793_B23GCKKLT3  False  False         True
1  127Cs-CMPN001_35_260114_LH00793_B23GCKKLT3  False  False         True
2  105Cc-CMPN001_23_260114_LH00793_B23GCKKLT3  False  False         True
3  070Ca-CMPN001_08_260114_LH00793_B23GCKKLT3  False  False         True
4  062Ca-CMPN001_06_260114_LH00793_B23GCKKLT3  False  False         True


In [10]:
# list of failed sample aliases

failed_aliases = no_run["alias"].tolist()
with open(filepath + "failed_aliases.txt", "w") as f:
    f.write("\n".join(failed_aliases))
print(len(failed_aliases), "aliases written to failed_aliases.txt")

27 aliases written to failed_aliases.txt
